<a href="https://colab.research.google.com/github/RRADJon/TEMPO/blob/main/NNFractionUnbound.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
#@title Single Fu Predictor Boltz-2 + NN Ensemble
!pip install -q rdkit
import os, requests, base64, time, json, torch, torch.nn as nn, io
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Crippen
from pathlib import Path
from google.colab import userdata

# --- 1. CONFIGURATION & INPUT ---
LIGAND_SMILES = "CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5"
NVIDIA_API_KEY = userdata.get('NVIDIA_API_KEY')
PUBLIC_URL = "https://health.api.nvidia.com/v1/biology/mit/boltz2/predict"
HEADERS = {"Authorization": f"Bearer {NVIDIA_API_KEY}", "Content-Type": "application/json"}

# Protein Data
PROTEINS = [
    {"id": "Albumin", "pdb": "1Ao6", "seq": "MKWVTFISLLFLFSSAYSRGVFRRDAHKSEVAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEVTEFAKTCVADESAENCDKSLHTLFGDKLCTVATLRETYGEMADCCAKQEPERNECFLQHKDDNPNLPRLVRPEVDVMCTAFHDNEETFLKKYLYEIARRHPYFYAPELLFFAKRYKAAFTECCQAADKAACLLPKLDELRDEGKASSAKQRLKCASLQKFGERAFKAWAVARLSQRFPKAEFAEVSKLVTDLTKVHTECCHGDLLECADDRADLAKYICENQDSISSKLKECCEKPLLEKSHCIAEVENDEMPADLPSLAADFVESKDVCKNYAEAKDVFLGMFLYEYARRHPDYSVVLLLRLAKTYETTLEKCCAAADPHECYAKVFDEFKPLVEEPQNLIKQNCELFEQLGEYKFQNALLVRYTKKVPQVSTPTLVEVSRNLGKVGSKCCKHPEAKRMPCAEDYLSVVLNQLCVLHEKTPVSDRVTKCCTESLVNRRPCFSALEVDETYVPKEFNAETFTFHADICTLSEKERQIKKQTALVELVKHKPKATKEQLKAVMDDFAAFVEKCCKADDKETCFAEEGKKLVAASQAALGL"},
    {"id": "AGP", "pdb": "3KQ0", "seq": "MALSWVLTVLSLLPLLEAQIPLCANLVPVPITNATLDRITGKWFYIASAFRNEEYNKSVQEIQATFFYFTPNKTEDTIFLREYQTRQDQCIYNTTYLNVQRENGTISRYVGGQEHFAHLLILRDTKTYMLAFDVNDEKNWGLSVYADKPETTKEQLGEFYEALDCLRIPKSDVVYTDWKKDKCEPLEKQHEKERKQEEGES"}
]

# --- 2. UTILITY FUNCTIONS ---
def get_physchem(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return 0.0, 0.0
    logP = Crippen.MolLogP(mol)
    fu_simple = np.clip(1 / (1 + (0.3 * 10**logP)), 0.001, 1.0)
    return logP, fu_simple

def get_template_b64(pdb_id):
    path = f"{pdb_id}.pdb"
    if not os.path.exists(path):
        r = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb")
        with open(path, "wb") as f: f.write(r.content)
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def predict_boltz(protein_id, sequence, template_b64, smiles):
    payload = {
        "polymers": [{
            "id": "A", "sequence": sequence, "molecule_type": "protein",
            "templates": [{"pdb": template_b64, "chain_id": "A"}]
        }],
        "ligands": [{"id": "L1", "smiles": smiles, "predict_affinity": True}]
    }
    resp = requests.post(PUBLIC_URL, headers=HEADERS, json=payload)
    if resp.status_code == 429:
        time.sleep(2); return predict_boltz(protein_id, sequence, template_b64, smiles)
    if resp.status_code != 200: raise Exception(f"API Error: {resp.text}")
    data = resp.json()
    aff = data.get("affinity_predictions", data.get("affinities", {})).get("L1", {})
    return aff.get("affinity_pic50", [0.0])[0], aff.get("affinity_probability_binary", [0.0])[0]

# --- 3. MODEL ARCHITECTURE & LOADING ---
class BinderNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.LayerNorm(32), nn.LeakyReLU(),
            nn.Linear(32, 16), nn.LeakyReLU(), nn.Linear(16, 1)
        )
    def forward(self, x): return self.net(x)

def load_ensemble(num_features=5):
    # Full Base64 String with Correct Padding
    MODEL_B64 = "UEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAWAAwAYmVzdF9mdV9tb2RlbC9kYXRhLnBrbEZCCABaWlpaWlpaWoACY2NvbGxlY3Rpb25zCk9yZGVyZWREaWN0CnEAKVJxAShYCAAAADAud2VpZ2h0cQJjdG9yY2guX3V0aWxzCl9yZWJ1aWxkX3RlbnNvcl92MgpxAygoWAcAAABzdG9yYWdlcQRjdG9yY2gKRmxvYXRTdG9yYWdlCnEFWAEAAAAwcQZYAwAAAGNwdXEHS6B0cQhRSwBLIEsFhnEJSwVLAYZxColoAClScQt0cQxScQ1YBgAAADAuYmlhc3EOaAMoKGgEaAVYAQAAADFxD2gHSyB0cRBRSwBLIIVxEUsBhXESiWgAKVJxE3RxFFJxFVgIAAAAMS53ZWlnaHRxFmgDKChoBGgFWAEAAAAycRdoB0sgdHEYUUsASyCFcRlLAYVxGoloAClScRt0cRxScR1YBgAAADEuYmlhc3EeaAMoKGgEaAVYAQAAADNxH2gHSyB0cSBRSwBLIIVxIUsBhXEiiWgAKVJxI3RxJFJxJVgIAAAAMy53ZWlnaHRxJmgDKChoBGgFWAEAAAA0cSdoB00AAnRxKFFLAEsQSyCGcSlLIEsBhnEqiWgAKVJxK3RxLFJxLVgGAAAAMy5iaWFzcS5oAygoaARoBVgBAAAANXEvaAdLEHRxMFFLAEsQhXExSwGFcTKJaAApUnEzdHE0UnE1WAgAAAA1LndlaWdodHE2aAMoKGgEaAVYAQAAADZxN2gHSxB0cThRSwBLAUsQhnE5SxBLAYZxOoloAClScTt0cTxScT1YBgAAADUuYmlhc3E+aAMoKGgEaAVYAQAAADdxP2gHSwF0cUBRSwBLAYVxQUsBhXFCiWgAKVJxQ3RxRFJxRXV9cUZYCQAAAF9tZXRhZGF0YXFHaAApUnFIKFgAAAAAcUl9cUpYBwAAAHZlcnNpb25xS0sBc1gBAAAAMHFMfXFNaEtLAXNYAQAAADFxTn1xT2hLSwFzWAEAAAAycVB9cVFoS0sBc1gBAAAAM3FSfXFTaEtLAXNYAQAAADRxVH1xVWhLSwFzWAEAAAA1cVZ9cVdoS0sBc3VzYi5QSwcI0jgljgoDAAAKAwAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAdACsAYmVzdF9mdV9tb2RlbC8uZm9ybWF0X3ZlcnNpb25GQicAWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaMVBLBwi379yDAQAAAAEAAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAACAAMQBiZXN0X2Z1X21vZGVsLy5zdG9yYWdlX2FsaWdubWVudEZCLQBaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlo2NFBLBwg/d3HpAgAAAAIAAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAABcAOQBiZXN0X2Z1X21vZGVsL2J5dGVvcmRlckZCNQBaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWmxpdHRsZVBLBwiFPeMZBgAAAAYAAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAABQAOABiZXN0X2Z1X21vZGVsL2RhdGEvMEZCNABaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpajT4nvt8siT70j2k+r8SIvpLeYD56lp6+ie6FvungE79YeGI+jvj6vuQpvj2/eQM+BUKXPsdfkD115Q2+L8YVPlUyxz6BZvI+xmKVvjfM4T7osKI+RfSAvjiIwb7xR0w+ueT2vkevpz7fYjw+dt4gvr1GAj8cQNk99IiGPv1k2D5iI80+Qs3IvH+fJL/Q9KE+eadcvXWb4b5VdUK+WymiPc12rD4YeYQ+Q2Lmva1W3j0kYhU9SmEQPSkibz1iYLK+zNuvvfIQWz5S++G+kZ/AvjM3tLrhhbs96DpXvr2KU755YI2+iyypPhnhwD2GvwC/vNHKvp8Akb5VMd6+5ewrP+ubBr6X+dE9GkP/PaauR76u6LA+DEmAPWEwor5+Z3G9RhIQPzds8j1yT6u+YisgPkB1gj4x3yQ9aVyRvWf5M79hVyo81iF5vTr6fr62p5u+YYQsPhw/Ur18tR2+n91AvJ93B77pAPw+V1U9vZYz6bxqRYC+E1gBPu9cGj9ynom+2j8WvAjfRD6tvle+8IHjPR8Bnz66ANw9Mo7OPPMiQ76AxXa7GwELvmEr7j2ckti+suy2vjnZLT4DtbA+qlC9Por7mz2XCss+GDDMvke6wj2uCBI+J7WqvXYwjT1BRuW9XD5ovmQ2zD6nO3E+7IbAPou02b5Vb+w+cRAoP0CzDDzheoM9X4ebPlcHOr4ODs6+Zz22Pa+3jj5czie8vbOHPFAHnj6RTX8+PFiIvuWMwz0IFhq/AsaIvjN7QL44vkI+rkQvvlBZz72b5tI8hlRqvsP2ML1Hxke9tquoPTUBkD6Y5ke/krHQvQ4phj5GWwq/UDckv91Rab4CcEE+Yfo0vVBLBwhiUhcbgAIAAIACAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAABQAPgBiZXN0X2Z1X21vZGVsL2RhdGEvMUZCOgBaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpa4iJSvJmcij722Rc9ep7XPW80bD4MGzW+jkjgvc7Pgj6i6Tm9e6Ynvv2Owzwo/HA+zn6FPlyX8Lzs4nw+n+4xPvDUZj1vRa4+hD6bPrA+k71YJ7o9f7hPvq6F4L6ss+s58ceZvgB3kL6/nZA9JQqBvfhPtD5NWy4+7e0wvuuF4T1QSwcI9jSGgIAAAACAAAAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAUAD4AYmVzdF9mdV9tb2RlbC9kYXRhLzJGQjoAWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWpLykT9RUa4/dKOoP2xikT+D9bg/ur66PwAquj+1g7Q/ofmkP3ZU7T8iHvk/pKmlP3zQ0T+gW54/BdrOPxRp1z++MsQ/Inm9P6venz/aDZw/rA+oP8oK8j/+0uU/7QWmP36u6D+21L8/HEi/P7QHpD9DPNw/CjmQPwkw6z9dR+E/UEsHCL7/nAuAAAAAgAAAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAAFAA+AGJlc3RfZnVfbW9kZWwvZGF0YS8zRkI6AFpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlo71EQ+6YFUvfcqXz7qJJu9j36nPqBEaj6IexE+T8PJvRYJgD5x7A6+Tb7VvaMMXT5+M3w+rkMVPjVlbT4KkUM+4EFzvsR7gzxB1Oc9eOwEvp3UkT26wNO8t+yGPJ/Ukr71rTq+S91bPqeTCz7fY8C+5f0QPqB6Wj7z0I098hqbvVBLBwj3KauYgAAAAIAAAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAABQAPgBiZXN0X2Z1X21vZGVsL2RhdGEvNEZCOgBaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpahvE2vhODaD43AzQ+FFucPaI0wL1UIEk+NIC1vX8/MT03aVc9inA4P6CUSrwTmAS+WlljvhAwLj5Wadg9TMjnPfZotzxLA+g9VcsYPp0lPL2/cIc8v1PjvgskHL7q/vA9sKVFPRwgdL5XTue9GnWePbg4YT3u2iU+UygZPvK1LL/fHr++K7RwPsqESb2IJgM+ue4yPir6hz4mDgS+Zpv5vk2MlD6yooE/cBlgP7ZVaT4zNjW+pxKnPlNykjwzNBk+5rfrPhC/27wlFqM+06CuvrXXa7zk7C+/yiLwvhC4Bz/gVwM/W39/vXB55bseqpY+D5U0vcVb8r0bopg+13wTvy/djbxoKPa9TiAEvSlepj0P7vu+sJyOvmEmyrz2dTI+kEgbPcI//b0s2jw+eEmFPSxfm7yzE+m9Qj2EvIN21jxi3Au+j3xiPXBkuTziF429vYmXPuUiB7+THKG+j4/YPTPXVD6l8J6+vc0qvutkSL5k+dG9dOw7vijV7z0C8nW91jqXvcYYWD62+ji++1lXPjtkLT6UHsI95MLvPfmazr3cSIs+jpEyPxQdZT86rMY9h0HOvbomZz6TLf48dTuuvAXFRj5yDgA8iUaCPnZfyj2kQCe+7QURvz6HZ77DZF89+ASqPtawhL70gwq+wBWSPrld971wclk93vp9PibwDr8BRfY+KDKPPswLlz5FoQk+dG4Cv0+DZb6eU48+0Q7EPsbem7xNEoG/IOYLvf/Sub4Cury+G7kGvUcoI74nVDc9IsjvvrnggL4CRgi/Go6bvirKCz9FFj4/Wx2NPitgyr6Nka0+Vg4YP8Pi0L3sGWS/bU0ZPuF21D6TrWy9M/EIPtgiiD0tXd+8vKt9vScjKL3Re3q9p45xvTb267yA07E9b6ubPfmtnj23FBS+eBQavrgBPb7GQZI9+mp/PeFqmb0BS7a9hroCvnzaIb4qheE9YNoVvpKzGr7y/JW8xSE9vANIK77BHB8+MiDBvOQ9GL6pAWc9a40ovguskz14ec28wxoZvOl4ND5Re7K9c3b2Pb63mzw8i3Y+mUOovG0wmL0j8Sg9cKEaP1k7XD/RTr69EECkvTXCiT1mcBa+dCT4PYPqgj0gPC8+yVCPPrBOXb7PmmC8XbDzvn3vB782RIg+mY5GPpaSBr469Ak98ymjPt0p4z2NONC9P7sDPs60CL+V+SA+kAt1PjTUDz47roK9zxervm992r4IWZo+bDoMvjH/Kb9FLMS+LCAXv/wzLL93g4s+R4UTPtKckj6JksG83lk8vtYiYb80mFm9m73LPomKHz5uhwm+4L1uPRsxRj3+Mnw+PM7yvQZYi78AsIq+uFeCPiGfKr3kVma/5n1YPiK8Jr3tSfY9eLPAPcEoQT4fpVk+9MeCPjlnv70Bg7O+RlA7PuwHiT54sCQ/aB6Mvf1llr1Ed8A9EK0jve5emT7KfZ89hVTwPTDRCTsI5oi+MsFmvkXXNL94T5i++SFPPTP20D4IC9a8LY8ZPqTlij7zOjo+0twSPQWgkTz7TPq+oai7PnW/mb4w9Jw+gdhcvoaXub6vRK2+iczcvarqRj6150K+fZ0wvy1IT7+N8Is9QnSZPveK3zxyk8Y+7Wb9vc/x0r2Cvz4+L8dDPgbwJT6UKDI+3cIvP0OT6j51SxC+CxEsPu77jz4pyzC8lifQvabTZT5jj9Q9wV6wvlLJtT4teAA+wUyhvkZPKL73T26+YrFnPr+cTL59Y74+ldjyPgUIsbty5Fa+qI3AvXEZxjswpAA/rDyUvYh27jyIaAM9ZTAgv72ozDznx3m86J/xvQw39D06LCO9q0+/PvgYjb1WdF+/KhzjPt30Hj3yes+9kPq4vqtXvD1nIM++L7LbPlsFlj60Ucq9G9s9vrt4zz7ecwm8r2gyv9Ovqj5fc2q9fQzsPUi5lT2eKwE/5+mnvrLOAb/zWQC/xpfxvau5iz6h5J0+cIJfvoPyXryp5Rw+iT8cvpnZar7D76y+nbgBPvXCBD6+eh8+w1Ocvq5x9z7GZ8K9PtIxvpwguz2ub5G+8KyRPjmO+74L7IY92yMHvo6CLb+ylpU+Vp8fP/7G+j4cIbi94mFhPXuVHj7aR9u988MlPx5qb710D2++YW2Kv58UgL+yPn69XqQKvZ83kz6j2M0+XzrXPdJfWz8QIBE9vnowvxVsyz7ml+s+17vYvRTGRL+UXTG/j8wpvmj6XT99c1E+NJ4FvgO8UT7JPro85m6mvpH7rjztRzw8eBj+Pb2gLb6OnZ++ESIfvry+U7zjnrQ927wRuxgxCT6bA0g8gap1vpy3ET74cuU95DO9Pe+pRj5p1cU+5PVqPrkI/DyLsjA+2XT5PVSrcj6TUZm+nDAHvuK+frwpj7S9E/LWPhf6jT6bfeQ8oOIjvKNeDT3MMBS+lSo8vsUCXz5Cn4I+HYTmve9/dL8Ry0i/4/x/vHHq4z5yNQu7cMt+PobSRz0+8QG+ix6nvV/fi70Du4k+e3V3PlP4GD/kAaQ+952KvUSiuL4vpqU+4L55vXkGhb7fPoW9vQCJPR4mx73FX/A+wdyDvZDnHD1iHiW/3xm9vS8HjD093/O9Dp/hPpcQWT5cDKc+kXOgvK+nvD7BuYc9CoUxv2dkVr3xsEq/myOoPsP/xb3bLKC+fYYzvm5XGL0ryig+IzLdvkP6hD4Sa528dPgRvoIPGz/HAJe+x2AYvsI5Sb4ds9E9yjCGvsp2Lj1QSwcIcZhywwAIAAAACAAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAUAD4AYmVzdF9mdV9tb2RlbC9kYXRhLzVGQjoAWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWqVtHz5Avg0+uqYHPu6OAz6ydc4+lQG/PQx0jT3jdnW9JVUtPd0a1L0T7Kw911SSvs8gCr6cyYo9nVuVvSR0Qr1QSwcI4Tb/w0AAAABAAAAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAUAD4AYmVzdF9mdV9tb2RlbC9kYXRhLzZGQjoAWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWqSxZr4PDhS/jhQ6v6Rnl77DS/s+qSEqPt/Z9745oYY/9Iz4vhzwiD7TTDo/Ib86v68bhD9g78I+Vh0tPrGeIT9QSwcIDrfazEAAAABAAAAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAUAD4AYmVzdF9mdV9tb2RlbC9kYXRhLzdGQjoAWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWphfMz5QSwcIQQdaCwQAAAAEAAAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAVADkAYmVzdF9mdV9tb2RlbC92ZXJzaW9uRkI1AFpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaMwpQSwcI0Z5nVQIAAAACAAAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAkACwAYmVzdF9mdV9tb2RlbC8uZGF0YS9zZXJpYWxpemF0aW9uX2lkRkIoAFpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlowODI2ODY0NzczNTYwNDU3NzcwMDE4MjkxOTc0MDg5ODI4MjAyMDI4UEsHCL6jwaUoAAAAKAAAAFBLAQIAAAAACAgAAAAAAADSOCWOCgMAAAoDAAAWAAAAAAAAAAAAAAAAAAAAAABiZXN0X2Z1X21vZGVsL2RhdGEucGtsUEsBAgAAAAAICAAAAAAAALfv3IMBAAAAAQAAAB0AAAAAAAAAAAAAAAAAWgMAAGJlc3RfZnVfbW9kZWwvLmZvcm1hdF92ZXJzaW9uUEsBAgAAAAAICAAAAAAAAD93cekCAAAAAgAAACAAAAAAAAAAAAAAAAAA0QMAAGJlc3RfZnVfbW9kZWwvLnN0b3JhZ2VfYWxpZ25tZW50UEsBAgAAAAAICAAAAAAAAIU94xkGAAAABgAAABcAAAAAAAAAAAAAAAAAUgQAAGJlc3RfZnVfbW9kZWwvYnl0ZW9yZGVyUEsBAgAAAAAICAAAAAAAAGJSFxuAAgAAgAIAABQAAAAAAAAAAAAAAAAA1gQAAGJlc3RfZnVfbW9kZWwvZGF0YS8wUEsBAgAAAAAICAAAAAAAAPY0hoCAAAAAgAAAABQAAAAAAAAAAAAAAAAA0AcAAGJlc3RfZnVfbW9kZWwvZGF0YS8xUEsBAgAAAAAICAAAAAAAAL7/nAuAAAAAgAAAABQAAAAAAAAAAAAAAAAA0AgAAGJlc3RfZnVfbW9kZWwvZGF0YS8yUEsBAgAAAAAICAAAAAAAAPcpq5iAAAAAgAAAABQAAAAAAAAAAAAAAAAA0AkAAGJlc3RfZnVfbW9kZWwvZGF0YS8zUEsBAgAAAAAICAAAAAAAAHGYcsMACAAAAAgAABQAAAAAAAAAAAAAAAAA0AoAAGJlc3RfZnVfbW9kZWwvZGF0YS80UEsBAgAAAAAICAAAAAAAAOE2/8NAAAAAQAAAABQAAAAAAAAAAAAAAAAAUBMAAGJlc3RfZnVfbW9kZWwvZGF0YS81UEsBAgAAAAAICAAAAAAAAA632sxAAAAAQAAAABQAAAAAAAAAAAAAAAAAEBQAAGJlc3RfZnVfbW9kZWwvZGF0YS82UEsBAgAAAAAICAAAAAAAAEEHWgsEAAAABAAAABQAAAAAAAAAAAAAAAAA0BQAAGJlc3RfZnVfbW9kZWwvZGF0YS83UEsBAgAAAAAICAAAAAAAANGeZ1UCAAAAAgAAABUAAAAAAAAAAAAAAAAAVBUAAGJlc3RfZnVfbW9kZWwvdmVyc2lvblBLAQIAAAAACAgAAAAAAAC+o8GlKAAAACgAAAAkAAAAAAAAAAAAAAAAANIVAABiZXN0X2Z1X21vZGVsLy5kYXRhL3NlcmlhbGl6YXRpb25faWRQSwYGLAAAAAAAAAAeAy0AAAAAAAAAAAAOAAAAAAAAAA4AAAAAAAAAxwMAAAAAAAB4FgAAAAAAAFBLBgcAAAAAPxoAAAAAAAABAAAAUEsFBgAAAAAOAA4AxwMAAHgWAAAAAA=="

    model_bytes = base64.b64decode(MODEL_B64 + "===") # Added safety padding
    buffer = io.BytesIO(model_bytes)
    model = BinderNet(num_features)
    state_dict = torch.load(buffer, map_location=torch.device('cpu'))

    first_key = list(state_dict.keys())[0]
    if not first_key.startswith("net."):
        model.net.load_state_dict(state_dict)
    else:
        model.load_state_dict(state_dict)
    model.eval()
    return model

# --- 4. THE PIPELINE ---
print(f"Starting PBPK Pipeline for: {LIGAND_SMILES[:30]}...")

logP, fu_simple = get_physchem(LIGAND_SMILES)
print(f"Calculated LogP: {logP:.2f}, Simple Fu: {fu_simple:.4f}")

results = {"logP": logP, "fu_simple": fu_simple}
for p in PROTEINS:
    print(f"Running Boltz-2 against {p['id']}...")
    b64 = get_template_b64(p['pdb'])
    pic50, prob = predict_boltz(p['id'], p['seq'], b64, LIGAND_SMILES)
    results[f"{p['id']}_pic50"] = pic50
    results[f"{p['id']}_prob"] = prob
    print(f"{p['id']} pIC50: {pic50:.2f}")

# Step 3: Ensemble Prediction & Calibration
features = ['logP', 'fu_simple', 'Albumin_pic50', 'Albumin_prob', 'AGP_pic50']
X_input = np.array([[results[f] for f in features]])

# Clipping inputs as per training (5th/95th percentiles)
X_input[:, 0] = np.clip(X_input[:, 0], -1.0, 6.5) # logP
X_input[:, 1] = np.clip(X_input[:, 1], 0.001, 1.0) # fu_simple

try:
    model = load_ensemble(num_features=len(features))
    X_tensor = torch.tensor(X_input, dtype=torch.float32)

    with torch.no_grad():
        # Predict in scaled log-space
        log_pred = model(X_tensor).item()
        # Inverse: y_trans = -np.log10(y_raw + 1e-5)
        raw_p = 10**(-log_pred) - 1e-5
        avg_p = np.clip(raw_p, 0, 1)

        # Power Calibration from training
        if avg_p > 0.5:
            final_fu = avg_p**0.8
        else:
            final_fu = avg_p**1.2

    final_fu = np.clip(final_fu, 0, 1)
    results["Predicted_fu"] = final_fu

    print(f"\n{'='*40}\nFINAL CALIBRATED PREDICTION (fu): {final_fu:.4f}\n{'='*40}")
    display(pd.DataFrame([results]))

except Exception as e:
    print(f"\n Prediction Failed: {e}")

Starting PBPK Pipeline for: CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C...
Calculated LogP: 4.59, Simple Fu: 0.0010
Running Boltz-2 against Albumin...
Albumin pIC50: 7.13
Running Boltz-2 against AGP...
AGP pIC50: 7.65

FINAL CALIBRATED PREDICTION (fu): 0.0180


,logP,fu_simple,Albumin_pic50,Albumin_prob,AGP_pic50,AGP_prob,Predicted_fu
0,4.59032,0.001,7.134359,0.625,7.651188,0.515625,0.017965
